# 04 — ML Improvement Experiments: Rare-Class Analysis, Robustness and Interpretability

Notebook 03 selected a KNN pipeline by group-aware cross-validation and evaluated it once on the held-out
test set. It reaches very high accuracy but recovers only a small fraction of the rare `reset-both` class.
This notebook investigates why and documents the trade-offs between alternatives.

**Rules:** training data only (`test.csv` is never loaded); same folds as Notebook 03; the KNN uses the deployed
hyperparameters from `models/metadata.json`; Notebook 03 is left unchanged; the overlap audit and strict group
CV already in Notebook 03 are cited, not repeated.

**Questions:** (1) precision/recall trade-off of the six classifiers on `reset-both`; (2) effect of class
weights; (3) where the missed rows go; (4) can a decision rule recover them; (5) which features carry the signal
and which exist only after the decision; (6) should the deployed model change?

## 1. Setup
Switches: `N_REPEATS` (permutation repeats), `RUN_EXTENSION` (extra notebook-local scenario without
`Elapsed Time (sec)`), `UPDATE_METRICS_JSON` (keep False until results are reviewed).

In [1]:
import sys, time, json, platform, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():          # allow running from notebooks/
    ROOT = ROOT.parent
assert (ROOT / "src").is_dir(), "Run from the repository root or from notebooks/."
sys.path.insert(0, str(ROOT))

from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import (classification_report, confusion_matrix,
                             balanced_accuracy_score, f1_score)
from sklearn.inspection import permutation_importance

import src.preprocessing as prep
from src.models import build_model, load_metadata, CLASS_LABELS, RANDOM_STATE
from src.preprocessing import get_feature_columns

REPORTS = ROOT / "reports"
REPORTS.mkdir(exist_ok=True)

N_REPEATS = 3
RUN_EXTENSION = True
UPDATE_METRICS_JSON = False

print("python", platform.python_version(), "| scikit-learn", sklearn.__version__,
      "| pandas", pd.__version__, "| numpy", np.__version__)
print("repository root:", ROOT.name)

python 3.14.4 | scikit-learn 1.9.1 | pandas 3.1.0 | numpy 2.5.4
repository root: Firewall-Traffic-Analyzer


## 2. Data, grouped folds and the deployed configuration
Group id = hash of the full 11-feature row (same as Notebook 03). `reset-both` has 43 training rows, so each
validation fold has 8–9; every fold must contain it. Sanity check: the mean macro-F1 must reproduce the `full`
row of `reports/feature_ablation.csv`; otherwise the folds differ and nothing below is comparable.

In [ ]:
train = pd.read_csv(ROOT / "data" / "processed" / "train.csv")   # test.csv is intentionally NOT loaded
FULL = get_feature_columns("full")
X_train, y_train = train[FULL], train["Action"]
assert len(FULL) == 11 and set(train.columns) == set(FULL) | {"Action"}

groups = pd.util.hash_pandas_object(X_train, index=False).to_numpy()
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
folds = list(cv.split(X_train, y_train, groups=groups))

for i, (tr, va) in enumerate(folds, 1):
    assert set(groups[tr]).isdisjoint(set(groups[va])), "group leakage between train/validation"
    counts = y_train.iloc[va].value_counts().reindex(CLASS_LABELS, fill_value=0).to_dict()
    print(f"fold {i}: validation rows={len(va):>5} | {counts}")
    assert counts["reset-both"] > 0, "a fold without reset-both"

meta = load_metadata()
DEPLOYED_PARAMS = meta["best_params"]
print("\ndeployed model:", meta["model_name"], "| feature_set:", meta["feature_set"], "|", DEPLOYED_PARAMS)

def deployed_knn(scenario="full"):
    model = build_model("KNN", scenario)
    model.set_params(**DEPLOYED_PARAMS)
    return model

def oof_predict(model, cols, method="predict"):
    """Out-of-fold predictions: every training row is predicted by a model that never saw it."""
    return cross_val_predict(model, X_train[cols], y_train, cv=folds, n_jobs=1, method=method)

def fold_means(pred):
    """Mean/std of per-fold macro-F1 and mean balanced accuracy (the Notebook-03 protocol)."""
    f1s, bas = [], []
    for _, va in folds:
        yt, yp = y_train.iloc[va], pred[va]
        f1s.append(f1_score(yt, yp, labels=CLASS_LABELS, average="macro", zero_division=0))
        bas.append(balanced_accuracy_score(yt, yp))
    return float(np.mean(f1s)), float(np.std(f1s)), float(np.mean(bas))

ref = pd.read_csv(REPORTS / "feature_ablation.csv").set_index("feature_set")
pred_deployed = oof_predict(deployed_knn(), FULL)
m_f1, s_f1, m_ba = fold_means(pred_deployed)
print(f"\nmean fold macro-F1 here = {m_f1:.6f} | Notebook 03 = {ref.loc['full', 'f1_macro_mean']:.6f}")
assert abs(m_f1 - ref.loc["full", "f1_macro_mean"]) < 0.002, "folds differ from Notebook 03"

fold 1: validation rows= 9148 | {'allow': 5990, 'deny': 1287, 'drop': 1862, 'reset-both': 9}
fold 2: validation rows= 9147 | {'allow': 5990, 'deny': 1287, 'drop': 1862, 'reset-both': 8}
fold 3: validation rows= 9147 | {'allow': 5990, 'deny': 1286, 'drop': 1862, 'reset-both': 9}
fold 4: validation rows= 9147 | {'allow': 5991, 'deny': 1287, 'drop': 1861, 'reset-both': 8}
fold 5: validation rows= 9147 | {'allow': 5990, 'deny': 1287, 'drop': 1861, 'reset-both': 9}

deployed model: KNN | feature_set: full | {'classifier__n_neighbors': 21, 'classifier__weights': 'distance'}
